# Satellite Chip Preparation
This notebook prepares the satellite imagery for model training. The input data consists 
of Sentinel-2 super-resolved images at 2.5m resolution, covering Brandenburg at three 
seasonal time points (April, June, August). The multitemporal setup allows the model to 
capture phenological patterns that distinguish flower strips from surrounding land use.

The preparation follows these steps:
1. Download: Super-resolved Sentinel-2 images are downloaded using rclone.
2. Reprojection: All images are reprojected to EPSG:3035. This is necessary because 
   Brandenburg spans two UTM zones (32 and 33), resulting in inconsistent coordinate 
   systems across the raw imagery.
3. VRT: A Virtual Raster (VRT) is built from all reprojected images, creating a seamless 
   mosaic without duplicating data on disk. A single-band export is provided for visual 
   inspection in QGIS.
4. Clip Chips: The VRT mosaic is clipped into chips of 592x592 pixels using template 
   chip boundaries as cookie-cutters. This ensures that satellite chips and label chips 
   are perfectly aligned.
5. Explore Chips: The chips are inspected to identify and document tiles containing 
   no-data values.

In [ ]:
# Import libraries
# to-do: Richtige Reihenfolge?
import os
from osgeo import gdal
from glob import glob
from multiprocessing import Pool
import time

import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.warp import Resampling, reproject

## 1. Download SR-Images

In [ ]:
# Set paths
# Section 2 - Reproject
image_dir        = '...'
reprojected_dir  = '...'
target_crs       = "EPSG:3035"

# Section 3 - Build VRT
vrt_file         = '...'
single_band_out  = '...'

In [ ]:
nrw_file = '...'
bb_file = '...'

for label, path in [("NRW", nrw_file), ("BB", bb_file)]:
    with rasterio.open(path) as src:
        arr = src.read()
        print(f"\n{label}: {src.count} bands")
        for i in range(src.count):
            b = arr[i]
            valid = b[np.isfinite(b) & (b != 0)]
            if valid.size:
                print(f"  band {i}: min={valid.min():.0f} "
                      f"median={np.median(valid):.0f} max={valid.max():.0f}")
            else:
                print(f"  band {i}: empty")

## 2. Reproject SR-Images 

In [ ]:
# create a list with the path to each image inside the image_dir folder
image_list = glob(os.path.join(image_dir, "*.tif"))

# loop through the list and reproject each image
for tif in image_list:
    out_tif = os.path.join(reprojected_dir, os.path.basename(tif))
    ds = gdal.Warp(out_tif, tif, dstSRS=target_crs, dstNodata=0)
    ds = None  # flush to disk

## 3. Build VRT

In [ ]:
# If you have already reprojected the images, you can start from here.Create another list with the reprojected files
reproj_list = glob(os.path.join(reprojected_dir, "*.tif"))

# Build VRT with absolute paths
vrt_options = gdal.BuildVRTOptions(
    resolution="average",
    addAlpha=True,
    srcNodata=0,
    separate=False,
)
vrt_ds = gdal.BuildVRT(vrt_file, reproj_list, options=vrt_options)

# Flush to disk>
vrt_ds = None

print(f"task done, vrt saved to {vrt_file}")

### 3.1. Single-Band VRT for Visual Inspection

In [ ]:
gdal.Translate(
    "output_band1_overviewaugustT.tif",
    vrt_file,
    bandList=[1],
    widthPct=10,  # 10% der originalen Auflösung
    heightPct=10,
    creationOptions=["COMPRESS=LZW"]
)

In [ ]:
start = time.time()
gdal.Translate(
    "output_band1APRIL.tif",
    vrt_file,
    bandList=[1],
    creationOptions=["COMPRESS=LZW", "BIGTIFF=YES"]
)

print(f"Dauer: {time.time() - start:.1f} Sekunden")

## 4. Clip Chips
This script processes a VRT file containing all Sentinel-2 super-resolved images by 
extracting chips at locations and sizes defined by template chips. Each output chip is 
resampled to a fixed 592x592 pixel size and saved as a GeoTIFF file. The script can be 
applied to both satellite imagery and rasterized flower strip label data.

Module Attributes:
    vrt_file (str): Path to the VRT mosaic file containing Sentinel-2 super-resolved images.
    CHIPS_DIR (str): Directory containing template chip GeoTIFF files used as cookie-cutters.
    OUT_DIR (str): Output directory where extracted chips will be saved.
    TARGET_SIZE (int): Target output chip dimensions (592x592 pixels).

Process:
    1. Opens the VRT file and reads its CRS, transform, band count, and data type.
    2. Iterates through all template chip files in CHIPS_DIR (sorted alphabetically).
    3. For each template chip:
       - Extracts its bounds and derives pixel resolution
       - Calculates a centered square region at TARGET_SIZE x TARGET_SIZE
       - Reprojects the VRT data to the target grid using bilinear resampling
       - Writes the extracted data as a GeoTIFF file with DEFLATE compression


In [ ]:
# to track progress: ls /workspaces/flowerstrips/data/flowerstripsdata/labels/output_cookie_cuts | wc -l
# set the path to the template chips, to the vrt, and to the output folder
# where the chips will be stored.
CHIPS_DIR = "..."
OUT_DIR = "..."
TARGET_SIZE = 592  # output must be 592 x 592. This is the optimal.
N_PROCESSES = 10    # adjust to your server

os.makedirs(OUT_DIR, exist_ok=True)

# list all the chips used as cookie-cutter
chip_files = sorted(glob(os.path.join(CHIPS_DIR, "*.tif")))  # limit to 100 for testing, remove [:100] for full run
def process_chip(chip_fp):
    out_name = os.path.splitext(os.path.basename(chip_fp))[0] + "_vrt_clip.tif"
    out_fp = os.path.join(OUT_DIR, out_name)
    
    if os.path.exists(out_fp):
        return out_name

    # open the vrt inside the function so multiprocessing works correctly
    with rasterio.open(vrt_file) as src:
        src_crs = src.crs
        src_transform = src.transform
        bands = src.count
        dtype = src.dtypes[0]
        src_nodata = src.nodata if src.nodata is not None else 0

        with rasterio.open(chip_fp) as chip:
            # derive pixel size from chip and center the square on chip extent
            xmin, ymin, xmax, ymax = chip.bounds
            resx = abs(chip.transform.a)
            resy = abs(chip.transform.e)
            res = (resx + resy) / 2.0
            cx = (xmin + xmax) / 2.0
            cy = (ymin + ymax) / 2.0
            half = (TARGET_SIZE * res) / 2.0
            new_xmin, new_xmax = cx - half, cx + half
            new_ymin, new_ymax = cy - half, cy + half
            dst_transform = from_bounds(
                new_xmin, new_ymin, new_xmax, new_ymax, TARGET_SIZE, TARGET_SIZE
            )

            # allocate destination filled with nodata
            dst_arr = np.full((bands, TARGET_SIZE, TARGET_SIZE), src_nodata, dtype=dtype)

            # reproject VRT -> target grid
            for b in range(bands):
                reproject(
                    source=rasterio.band(src, b + 1),
                    destination=dst_arr[b],
                    src_transform=src_transform,
                    src_crs=src_crs,
                    dst_transform=dst_transform,
                    dst_crs=chip.crs,
                    resampling=Resampling.nearest, # change here to resampling=Resampling.nearest (instead of bilinear) for rasterized labels
                    dst_nodata=src_nodata,
                )

            # write output (preserve chip CRS and updated size/transform)
            out_profile = chip.profile.copy()
            out_profile.update({
                "height": TARGET_SIZE,
                "width": TARGET_SIZE,
                "transform": dst_transform,
                "crs": chip.crs,
                "count": bands,
                "dtype": dtype,
                "compress": "DEFLATE",
            })

            out_name = os.path.splitext(os.path.basename(chip_fp))[0] + "_vrt_clip.tif"
            out_fp = os.path.join(OUT_DIR, out_name)

            with rasterio.open(out_fp, "w", **out_profile) as dst:
                dst.write(dst_arr)

    return out_name


if __name__ == "__main__":
    with Pool(N_PROCESSES) as pool:
        pool.map(process_chip, chip_files)

    print("Done. Outputs in:", OUT_DIR)

## 5. Explore Chips & Identify No-Data
Keep only Sentinel-2 SR chips that,
1) exist in all required month folders
2) are not empty

The script copies only valid chips into a new output directory,
keeping one subfolder per month.

In [ ]:
from pathlib import Path
import shutil
import rasterio
import numpy as np

# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

MONTH_DIRS = {
    "april": Path(r"..."),
    "june": Path(r"..."),
    "august": Path(r"..."),
}

OUT_BASE = Path(r"...")

FILE_EXTENSION = "*.tif"

# --------------------------------------------------
# FUNCTIONS
# --------------------------------------------------

def get_chip_id(filepath: Path) -> str:
    return filepath.stem

# This code discards all chips where a band is ‘NoData’ – in other words, not where perhaps 30 per cent of the chip’s pixels are ‘NoData’, but where an entire band is completely ‘NoData’!
def is_empty_chip(raster_path):
    """
    Returns True if the chip is unusable.
    A chip is considered unusable if ANY band is:
    - fully masked / nodata
    - all NaN
    - all zero
    - has no finite valid values
    """
    with rasterio.open(raster_path) as src:
        data = src.read(masked=True)

        for b in range(data.shape[0]):
            band = data[b]

            # all masked
            if np.ma.count(band) == 0:
                return True

            # valid unmasked values
            valid = band.compressed()

            if valid.size == 0:
                return True

            # keep only finite values
            valid = valid[np.isfinite(valid)]

            if valid.size == 0:
                return True

            # all zero
            if np.all(valid == 0):
                return True

        return False


def collect_files(folder: Path) -> dict:
    files = sorted(folder.glob(FILE_EXTENSION))
    return {get_chip_id(fp): fp for fp in files}


# --------------------------------------------------
# MAIN
# --------------------------------------------------

def main():
    print("Collecting chip files...")

    month_files = {}
    for month, folder in MONTH_DIRS.items():
        if not folder.exists():
            raise FileNotFoundError(f"Folder not found: {folder}")
        month_files[month] = collect_files(folder)
        print(f"{month}: {len(month_files[month])} files found")

    common_ids = set.intersection(*(set(d.keys()) for d in month_files.values()))
    print(f"\nCommon chip IDs across all months: {len(common_ids)}")

    valid_ids = []
    invalid_missing_or_empty = []

    print("\nChecking whether common chips are empty...")
    for chip_id in sorted(common_ids):
        keep_chip = True

        for month, files_dict in month_files.items():
            chip_path = files_dict[chip_id]

            if is_empty_chip(chip_path):
                keep_chip = False
                invalid_missing_or_empty.append((chip_id, month, chip_path.name))
                print(f"Removed: {chip_id} | month: {month} | file: {chip_path.name}")
                break

        if keep_chip:
            valid_ids.append(chip_id)

    print(f"\nInvalid chip IDs removed: {len(invalid_missing_or_empty)}")
    print(f"Valid chip IDs present and non-empty in all months: {len(valid_ids)}")

    for month in MONTH_DIRS.keys():
        (OUT_BASE / month).mkdir(parents=True, exist_ok=True)

    print("\nCopying valid chips...")
    for chip_id in valid_ids:
        for month, files_dict in month_files.items():
            src = files_dict[chip_id]
            dst = OUT_BASE / month / src.name
            shutil.copy2(src, dst)

    print("\nDone.")
    print(f"Filtered chips saved to: {OUT_BASE}")

    valid_ids_txt = OUT_BASE / "valid_chip_ids.txt"
    with open(valid_ids_txt, "w", encoding="utf-8") as f:
        for chip_id in valid_ids:
            f.write(chip_id + "\n")

    print(f"Valid chip ID list saved to: {valid_ids_txt}")


if __name__ == "__main__":
    main()

In [ ]:
# Creates a footprint layer!
import rasterio
import geopandas as gpd
from shapely.geometry import box
from glob import glob

files = glob('...')

rows = []
for f in files:
    with rasterio.open(f) as src:
        b = src.bounds
        rows.append({
            'filename': f.split('/')[-1],
            'geometry': box(b.left, b.bottom, b.right, b.top)
        })

gdf = gpd.GeoDataFrame(rows, crs='EPSG:3035')
gdf.to_file('.../SR-Chips_footprintNRW.gpkg')
print(f'{len(gdf)} Data stored as a footprint')

In [ ]:
# Nach diesem Schritt go back to Label-Chip Preparation um die Chips bei den Labels auch auszusortieren die außerhalb des Bundeslands liegen
import geopandas as gpd
import shutil
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────'
gpkg_fp      = '.../SR-Chips_footprint_outsideNRW.gpkg'
filtered_dir = Path('...')
outside_dir  = Path('...')
months       = ['april', 'june', 'august']
# ─────────────────────────────────────────────────────────────────────────────

# load outside BB chip names
gdf = gpd.read_file(gpkg_fp)
outside_ids = set(gdf['filename'].tolist())
print(f"Chips to move: {len(outside_ids)}")

for month in months:
    src_dir = filtered_dir / month
    dst_dir = outside_dir / month
    dst_dir.mkdir(parents=True, exist_ok=True)

    moved = 0
    for chip_id in outside_ids:
        src = src_dir / chip_id
        if src.exists():
            shutil.move(str(src), str(dst_dir / chip_id))
            moved += 1

    print(f"{month}: {moved} chips moved")

print(f"\nDone. Moved to: {outside_dir}")